# Common held-out evaluation

In [11]:
from pathlib import Path
import sys, json
import torch, torchvision
project_dir = Path.home() / "t2-project"
sys.path.insert(0, str(project_dir))
from t2.train import train, load_policy
from t2.data import preparation, vocabulary, Demonstrations
manifest, normalization, manifest_hash = preparation(project_dir)
print("Project:", project_dir)

Project: /home/njmcmil/t2-project


## 1. Declare a target and select common states


In [12]:
from t2.simulation import prepare_protocol
success_target = 0.50
target_justification = "Pilot project target: complete at least half of held-out rollouts on average across the two tasks; fixed before final testing."
protocol_path = prepare_protocol(project_dir, success_target=success_target)
print(protocol_path)

[info] using task orders [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
[info] using task orders [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
/home/njmcmil/t2-project/configs/evaluation_protocol.json


## 2. Choose checkpoints using validation only
Find the lowest validation loss among runs for each variant/seed. 

In [13]:
candidates = {}
for path in (project_dir / "runs").glob("*/best_model.pt"):
    saved = torch.load(path, map_location="cpu", weights_only=True)
    config = saved["config"]
    if config.get("architecture") != "compact_attention_v1" or config.get("smoke_test"):
        continue
    key = (config["variant"], config["training_seed"])
    if key not in candidates or saved["validation_mse"] < candidates[key][0]:
        candidates[key] = (saved["validation_mse"], path)
for key, (loss, path) in sorted(candidates.items()):
    print(key, loss, path)

('dense', 0) 0.619675495972236 /home/njmcmil/t2-project/runs/dense_seed0_20261117T211545_858127Z/best_model.pt
('dense', 1) 0.6421160671032138 /home/njmcmil/t2-project/runs/dense_seed1_20261117T211813_215453Z/best_model.pt
('grids', 0) 0.5080180598629845 /home/njmcmil/t2-project/runs/grids_seed0_20261117T212203_540220Z/best_model.pt
('grids', 1) 0.5040738909608788 /home/njmcmil/t2-project/runs/grids_seed1_20261117T212424_721238Z/best_model.pt
('grids_film', 0) 0.517944386228919 /home/njmcmil/t2-project/runs/grids_film_seed0_20261117T212845_384875Z/best_model.pt
('grids_film', 1) 0.613483181844155 /home/njmcmil/t2-project/runs/grids_film_seed1_20261117T212956_979682Z/best_model.pt
('uniform', 0) 0.5764545529252953 /home/njmcmil/t2-project/runs/uniform_seed0_20261117T212318_683813Z/best_model.pt
('uniform', 1) 0.6289980589515634 /home/njmcmil/t2-project/runs/uniform_seed1_20261117T212534_924986Z/best_model.pt
('vision_only', 0) 0.6162079601652093 /home/njmcmil/t2-project/runs/vision_only

## 3. Freeze the selection


In [14]:
from t2.evaluate import freeze_selection
selection_path = freeze_selection(project_dir, [item[1] for item in candidates.values()], success_target, target_justification)
print(selection_path)

FileExistsError: Final selection already frozen. Do not overwrite after viewing test outcomes.

## 4. Run final evaluation
This is 400 rollouts and writes videos and results. Set RUN_FINAL to True only when model selection is complete. Completed cases are checked and skipped on restart; failed executions stop rather than silently dropping trials.

In [15]:
from t2.evaluate import evaluate_selected
RUN_FINAL = True
if RUN_FINAL:
    evaluate_selected(project_dir)
else:
    print("Final evaluation has not been started.")

[info] using task orders [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
put_the_bowl_on_the_plate put_the_bowl_on_the_plate_benchmark_32 success= False
[info] using task orders [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
put_the_bowl_on_the_plate put_the_bowl_on_the_plate_benchmark_45 success= False
[info] using task orders [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
put_the_bowl_on_the_plate put_the_bowl_on_the_plate_benchmark_30 success= False
[info] using task orders [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
put_the_bowl_on_the_plate put_the_bowl_on_the_plate_benchmark_46 success= False
[info] using task orders [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
put_the_bowl_on_the_plate put_the_bowl_on_the_plate_benchmark_3 success= False
[info] using task orders [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
put_the_bowl_on_the_plate put_the_bowl_on_the_plate_benchmark_18 success= False
[info] using task orders [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
put_the_bowl_on_the_plate put_the_bowl_on_the_plate_benchmark_36 success= False
[info] using task orders [0, 1, 2, 3, 4, 5, 6, 7,

## 5. Build the comparison table
Incomplete evaluations remain labeled incomplete. Success rates use 20 rollouts per task/seed. Latency is batch-one model-only inference after warmup; it excludes camera rendering and preprocessing. Two training seeds provide limited variability evidence.

In [16]:
from t2.results import summarize
rows = summarize(project_dir)
for row in rows:
    print(row)

[
  {
    "variant": "dense",
    "status": "complete",
    "mean_success_rate": 0.037500000000000006,
    "seed_rates": [
      0.025,
      0.05
    ],
    "between_seed_sample_std": 0.01767766952966369,
    "success_target": 0.5,
    "target_met": false,
    "latency_target_met": true
  },
  {
    "variant": "grids",
    "status": "complete",
    "mean_success_rate": 0.1,
    "seed_rates": [
      0.15,
      0.05
    ],
    "between_seed_sample_std": 0.07071067811865475,
    "success_target": 0.5,
    "target_met": false,
    "latency_target_met": true
  },
  {
    "variant": "grids_film",
    "status": "complete",
    "mean_success_rate": 0.037500000000000006,
    "seed_rates": [
      0.05,
      0.025
    ],
    "between_seed_sample_std": 0.01767766952966369,
    "success_target": 0.5,
    "target_met": false,
    "latency_target_met": true
  },
  {
    "variant": "uniform",
    "status": "complete",
    "mean_success_rate": 0.125,
    "seed_rates": [
      0.15,
      0.1
    ]